<a href="https://colab.research.google.com/github/ankurdev1-drth/Deep_Learning-/blob/main/02_Training_Deep_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [1]:
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import torch

layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5  # kaiming init (or 3 ** 0.5 for LeCun)
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [3]:
layer.weight.shape

torch.Size([10, 40])

In [4]:
layer.bias.shape

torch.Size([10])

In [5]:
# another way to for initialization
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

This is clearer and less error prone as compared to the first approach

For applying initialization to the weights of every `nn.Linear` layer in a model the simples option is to write a little function that takes a module, checks whether it's an instance of the `nn.Linear` class, and if so , applies the desired initialization function to its weights. and the function can be then applied using the `apply()` method

In [6]:
def use_he_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    nn.init.zeros_(module.bias)

module = nn.Sequential(
    nn.Linear(40, 10),
    nn.ReLU(),
    nn.Linear(10, 1),
    nn.ReLU()
    )
module.apply(use_he_init)

Sequential(
  (0): Linear(in_features=40, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=1, bias=True)
  (3): ReLU()
)

### Leaky ReLU

In [7]:
alpha = 0.2
model = nn.Sequential(
    nn.Linear(50, 40), #0
    nn.LeakyReLU(negative_slope=alpha), #1
    nn.Linear(40, 20), #2
    nn.LeakyReLU(negative_slope=alpha) #3

)
nn.init.kaiming_uniform_(model[0].weight, alpha,
nonlinearity="leaky_relu")
nn.init.kaiming_uniform_(model[2].weight, alpha,nonlinearity="leaky_relu")

Parameter containing:
tensor([[-0.3476,  0.2175, -0.1756, -0.3724,  0.1879,  0.1180,  0.1525,  0.0443,
         -0.0695,  0.1737, -0.2772, -0.2514,  0.1721,  0.1085,  0.0201,  0.1234,
          0.3707, -0.1772,  0.1891, -0.1664,  0.2776,  0.1537, -0.0671,  0.2200,
         -0.1501,  0.0461, -0.0360, -0.1530, -0.1844,  0.2199, -0.0969, -0.0314,
         -0.1147, -0.0029, -0.3225, -0.0798,  0.3271, -0.0053, -0.3348,  0.2127],
        [-0.3622,  0.0582, -0.1054, -0.1839, -0.0958,  0.1104,  0.2067,  0.3259,
         -0.2312,  0.2490, -0.2225,  0.1429, -0.0974, -0.2343, -0.1857, -0.2315,
         -0.2846,  0.1988, -0.0287, -0.0430, -0.3083, -0.0750, -0.2374,  0.1369,
         -0.1027, -0.1335,  0.1883, -0.1794, -0.1330, -0.2006, -0.1418, -0.1326,
         -0.3056,  0.2681, -0.0485,  0.0560, -0.1552, -0.0429, -0.1771, -0.3729],
        [-0.3518, -0.1798, -0.1160,  0.3745, -0.2538, -0.0830,  0.1376,  0.1816,
          0.0650, -0.1829, -0.0977,  0.1927, -0.3322,  0.3772,  0.2637,  0.0905,
    

### Batch Normalization

In [8]:
model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(1 * 28 * 28),
    nn.Linear(1 * 28 * 28,  300),
    nn.ReLU(),
    nn.BatchNorm1d(300),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.BatchNorm1d(100),
    nn.Linear(100, 10)
)

In [9]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [10]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [11]:
dict(model[1].named_children()).keys()

dict_keys([])

In [12]:
dict(model[1].named_modules()).keys()

dict_keys([''])

Note:
- if BN layers before the activation functions, we can also remove the bias term from the previous `nn.Linear` layers by setting the `bias` hyperparameter to `False`.

In [13]:
Model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300, 100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

### Layer Normalization

In [14]:
inputs = torch.randn(32, 3, 100, 200)   # batch of random RGB images
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs)
result

tensor([[[[-2.0521e-01, -1.7464e-01, -2.5041e-01,  ..., -7.9921e-01,
            1.1750e+00,  1.7518e+00],
          [ 8.3065e-01, -5.6911e-01,  1.0079e-01,  ...,  1.0604e+00,
           -2.3064e-01, -3.2774e-01],
          [-4.2386e-01, -1.4343e-01,  8.5209e-01,  ..., -3.9762e-01,
            6.3441e-01,  1.6126e+00],
          ...,
          [ 8.2832e-01,  6.6922e-01,  2.5394e-01,  ...,  5.5879e-01,
            2.8824e-01,  1.2999e+00],
          [ 1.0477e+00, -1.7391e+00,  1.8853e+00,  ...,  3.5820e-01,
            1.5440e-01,  9.0162e-01],
          [ 8.9041e-01, -6.0673e-01, -1.5907e+00,  ...,  4.8923e-01,
            1.3932e+00,  4.3722e-01]],

         [[ 4.6623e-01, -6.4669e-01,  6.2430e-01,  ..., -9.1497e-01,
           -1.7740e-01, -8.8824e-02],
          [ 2.0744e-01, -3.3211e-01, -1.1081e+00,  ..., -1.5624e-01,
           -2.1282e-01, -5.7380e-01],
          [-7.6346e-01, -1.0987e+00, -9.7857e-01,  ...,  1.1039e+00,
           -7.1255e-01, -1.1831e+00],
          ...,
     

In [15]:
# method 2 to perform the same thing as above!
means = inputs.mean(dim=[2,3], keepdim=True)  #shape [32, 3, 1, 1]
vars_ = inputs.var(dim=[2,3], keepdim=True, unbiased=False) # shape: same
stds = torch.sqrt(vars_ + layer_norm.eps) # eps is a smoothing term (1e-5)
result = layer_norm.weight * (inputs - means) / stds + layer_norm.bias

In [16]:
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs)

In [17]:
result

tensor([[[[-2.1230e-01, -1.8150e-01, -2.5784e-01,  ..., -8.1072e-01,
            1.1782e+00,  1.7593e+00],
          [ 8.3127e-01, -5.7890e-01,  9.5975e-02,  ...,  1.0628e+00,
           -2.3792e-01, -3.3574e-01],
          [-4.3258e-01, -1.5005e-01,  8.5287e-01,  ..., -4.0614e-01,
            6.3357e-01,  1.6191e+00],
          ...,
          [ 8.2893e-01,  6.6864e-01,  2.5027e-01,  ...,  5.5739e-01,
            2.8482e-01,  1.3040e+00],
          [ 1.0499e+00, -1.7576e+00,  1.8938e+00,  ...,  3.5530e-01,
            1.4998e-01,  9.0276e-01],
          [ 8.9147e-01, -6.1681e-01, -1.6081e+00,  ...,  4.8731e-01,
            1.3980e+00,  4.3491e-01]],

         [[ 4.8569e-01, -6.2375e-01,  6.4327e-01,  ..., -8.9118e-01,
           -1.5592e-01, -6.7623e-02],
          [ 2.2772e-01, -3.1015e-01, -1.0838e+00,  ..., -1.3483e-01,
           -1.9123e-01, -5.5108e-01],
          [-7.4015e-01, -1.0743e+00, -9.5459e-01,  ...,  1.1214e+00,
           -6.8940e-01, -1.1585e+00],
          ...,
     

## Gradient Clipping

See the line nn.utils.clip_grad_norm_(...) in the training function in the next section.

# Reusing Pretrained Layers

### Transfer Learning with PyTorch

- X_train_A = images of all items except for T-shirts/tops and pullovers (classes 0 and 2 )
- X_train_B = first 20 images of Tshirt and Pullovers

The validation set and the test set are also split this way, but without restricting the number of images.

We will train a model on set A (classification task with 8 classes), and try to reuse it to tackle set B (binary classification). We hope to transfer a little bit of knowledge from task A to task B, since classes in set A (trousers, dresses, coats, sandals, shirts, sneakers, bags, and ankle boots) are somewhat similar to classes in set B (T-shirts/tops and pullovers). However, since we are using Linear layers, only patterns that occur at the same location can be reused (in contrast, convolutional layers will transfer much better, since learned patterns can be detected anywhere on the image)


In [18]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_openml


fashion_mnist = fetch_openml(name="Fashion-MNIST", as_frame=False) #as_frames = False for getting the numpy arrays instead of pandas dataframe/series
X = torch.FloatTensor(fashion_mnist.data.reshape(-1, 1, 28, 28) / 255.)
y = torch.from_numpy(fashion_mnist.target.astype(int))
in_B = (y == 0) | (y == 2)  # Pullover or T-shirt/top
X_A, y_A = X[~in_B], y[~in_B]
y_A = torch.maximum(y_A - 2, torch.tensor(0))  # [1,3,4,5,6,7,8,9] => [0,..,7]
X_B, y_B = X[in_B], (y[in_B] == 2).to(dtype=torch.float32).view(-1, 1)

train_set_A = TensorDataset(X_A[:-7_000], y_A[:-7000])
valid_set_A = TensorDataset(X_A[-7_000:-5000], y_A[-7000:-5000])
test_set_A = TensorDataset(X_A[-5_000:], y_A[-5000:])
train_set_B = TensorDataset(X_B[:20], y_B[:20])
valid_set_B = TensorDataset(X_B[20:5000], y_B[20:5000])
test_set_B = TensorDataset(X_B[5_000:], y_B[5000:])

train_loader_A = DataLoader(train_set_A, batch_size=32, shuffle=True)
valid_loader_A = DataLoader(valid_set_A, batch_size=32)
test_loader_A = DataLoader(test_set_A, batch_size=32)
train_loader_B = DataLoader(train_set_B, batch_size=32, shuffle=True)
valid_loader_B = DataLoader(valid_set_B, batch_size=32)
test_loader_B = DataLoader(test_set_B, batch_size=32)

In [19]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device


'cpu'

In [20]:
%pip install torchmetrics


import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            # Uncomment to activate gradient clipping:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 17.0 MB/s eta 0:00:00


In [21]:
torch.manual_seed(42)

model_A = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 8)
  )
model_A = model_A.to(device)

In [22]:
model_A.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=8, bias=True)
)

In [23]:
n_epochs = 20
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.005)
loss_fn = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)
history_A = train(model_A, optimizer, loss_fn, accuracy, train_loader_A, valid_loader_A, n_epochs )

Epoch 1/20, train loss: 0.8492, train metric: 0.7197, valid metric: 0.8120
Epoch 2/20, train loss: 0.4533, train metric: 0.8487, valid metric: 0.8620
Epoch 3/20, train loss: 0.3771, train metric: 0.8727, valid metric: 0.8685
Epoch 4/20, train loss: 0.3405, train metric: 0.8828, valid metric: 0.8890
Epoch 5/20, train loss: 0.3184, train metric: 0.8904, valid metric: 0.8825
Epoch 6/20, train loss: 0.3033, train metric: 0.8953, valid metric: 0.8875
Epoch 7/20, train loss: 0.2911, train metric: 0.9000, valid metric: 0.8900
Epoch 8/20, train loss: 0.2818, train metric: 0.9031, valid metric: 0.8960
Epoch 9/20, train loss: 0.2737, train metric: 0.9053, valid metric: 0.8985
Epoch 10/20, train loss: 0.2668, train metric: 0.9082, valid metric: 0.8900
Epoch 11/20, train loss: 0.2611, train metric: 0.9105, valid metric: 0.9035
Epoch 12/20, train loss: 0.2547, train metric: 0.9125, valid metric: 0.8985
Epoch 13/20, train loss: 0.2502, train metric: 0.9145, valid metric: 0.8855
Epoch 14/20, train lo

In [24]:
torch.manual_seed(42)
model_B = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 1)
).to(device)

In [25]:
model_B.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=1, bias=True)
)

In [26]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B, optimizer, loss_fn, accuracy, train_loader_B, valid_loader_B, n_epochs)

Epoch 1/20, train loss: 0.7110, train metric: 0.5000, valid metric: 0.4914
Epoch 2/20, train loss: 0.7083, train metric: 0.5000, valid metric: 0.4954
Epoch 3/20, train loss: 0.7055, train metric: 0.5000, valid metric: 0.4976
Epoch 4/20, train loss: 0.7028, train metric: 0.5000, valid metric: 0.5008
Epoch 5/20, train loss: 0.7001, train metric: 0.5000, valid metric: 0.5034
Epoch 6/20, train loss: 0.6974, train metric: 0.5000, valid metric: 0.5080
Epoch 7/20, train loss: 0.6947, train metric: 0.5000, valid metric: 0.5118
Epoch 8/20, train loss: 0.6920, train metric: 0.5000, valid metric: 0.5159
Epoch 9/20, train loss: 0.6893, train metric: 0.5000, valid metric: 0.5197
Epoch 10/20, train loss: 0.6866, train metric: 0.5000, valid metric: 0.5225
Epoch 11/20, train loss: 0.6840, train metric: 0.5000, valid metric: 0.5263
Epoch 12/20, train loss: 0.6813, train metric: 0.5000, valid metric: 0.5311
Epoch 13/20, train loss: 0.6787, train metric: 0.5000, valid metric: 0.5339
Epoch 14/20, train lo

In [27]:
# reusing all the layers except the output layer
import copy

torch.manual_seed(42)
reused_layer = copy.deepcopy(model_A[: -1])
model_B_on_A = nn.Sequential(
    *reused_layer,
    nn.Linear(100, 1) # new output layer for task B
).to(device)

Note:
- `copy.deepcopy()` function to copy all the modules and submodules in `nn.Sequential` module
- model_B_on_A is another `nn.Sequential` module based on the reused layers of model A plus a new output layer for task B it has a single output since task B is binary classification.

In [28]:
# freezing the reused layers during the first few epochs:
for layer in model_B_on_A[:-1]:
  for param in layer.parameters():
    param.requires_grad = False


In [29]:
n_epochs = 10
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs)

Epoch 1/10, train loss: 0.6149, train metric: 0.4000, valid metric: 0.5402
Epoch 2/10, train loss: 0.5850, train metric: 0.4500, valid metric: 0.5783
Epoch 3/10, train loss: 0.5589, train metric: 0.6500, valid metric: 0.6410
Epoch 4/10, train loss: 0.5367, train metric: 0.8000, valid metric: 0.7179
Epoch 5/10, train loss: 0.5185, train metric: 0.8500, valid metric: 0.7855
Epoch 6/10, train loss: 0.5041, train metric: 0.8500, valid metric: 0.8367
Epoch 7/10, train loss: 0.4933, train metric: 0.9000, valid metric: 0.8663
Epoch 8/10, train loss: 0.4851, train metric: 0.9000, valid metric: 0.8787
Epoch 9/10, train loss: 0.4783, train metric: 0.8500, valid metric: 0.8855
Epoch 10/10, train loss: 0.4718, train metric: 0.8500, valid metric: 0.8898


In [30]:
# unfreezing the reused layers during the first few epochs:
for layer in model_B_on_A[2:]:
  for param in layer.parameters():
    param.requires_grad = True

In [31]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.01)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                train_loader_B, valid_loader_B, n_epochs)

Epoch 1/20, train loss: 0.4654, train metric: 0.8500, valid metric: 0.8972
Epoch 2/20, train loss: 0.4511, train metric: 0.9000, valid metric: 0.9022
Epoch 3/20, train loss: 0.4372, train metric: 0.9000, valid metric: 0.9074
Epoch 4/20, train loss: 0.4238, train metric: 0.9500, valid metric: 0.9112
Epoch 5/20, train loss: 0.4108, train metric: 0.9500, valid metric: 0.9139
Epoch 6/20, train loss: 0.3983, train metric: 0.9500, valid metric: 0.9173
Epoch 7/20, train loss: 0.3861, train metric: 0.9500, valid metric: 0.9179
Epoch 8/20, train loss: 0.3744, train metric: 0.9500, valid metric: 0.9193
Epoch 9/20, train loss: 0.3631, train metric: 0.9500, valid metric: 0.9213
Epoch 10/20, train loss: 0.3521, train metric: 0.9500, valid metric: 0.9221
Epoch 11/20, train loss: 0.3415, train metric: 0.9500, valid metric: 0.9233
Epoch 12/20, train loss: 0.3312, train metric: 0.9500, valid metric: 0.9249
Epoch 13/20, train loss: 0.3212, train metric: 0.9500, valid metric: 0.9259
Epoch 14/20, train lo

In [32]:
evaluate_tm(model_B_on_A, test_loader_B, accuracy)

tensor(0.9360)

## Faster Optimizers

In [33]:
# to test an optimizer on fashion MNIST :
train_set = TensorDataset(X[:55_000], y[:55_000])
valid_set = TensorDataset(X[55_000:60_000], y[55_000:60_000])
test_set = TensorDataset(X[60_000:], y[60_000:])

def build_model(seed=43):
  torch.manual_seed(seed)
  model = nn.Sequential(
      nn.Flatten(),
      nn.Linear(28 * 28, 100),
      nn.ReLU(),
      nn.Linear(100, 100),
      nn.ReLU(),
      nn.Linear(100, 100),
      nn.ReLU(),
      nn.Linear(100, 10)
  ).to(device)
  model.apply(use_he_init)
  return model

def test_optimizer(model, optimizer, n_epochs=10, batch_size=32):
  train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
  valid_loader = DataLoader(valid_set, batch_size=batch_size)
  test_loader = DataLoader(test_set, batch_size=32)
  xentropy = nn.CrossEntropyLoss()
  accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
  history = train(model, optimizer, xentropy, accuracy.to(device),
                  train_loader, valid_loader, test_loader)
  return history, evaluate_tm(model, test_loader, accuracy)
